# WorldSim — Training Notebook

Train a Convolutional RSSM world model on your own video data.

**Before running:**
1. Add your dataset as input (the folder with `frame_XXXXX.jpg` + `dataset.jsonl`)
2. Update `BASE` path below to match your dataset
3. Enable GPU accelerator (T4)
4. Run All

**Output:** `worldmodel_v2.pt` + `trajectory.gif` in `/kaggle/working/`

In [ ]:
import os, json
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

In [ ]:
# ======================================================
# UPDATE THIS PATH to match your Kaggle dataset
BASE = Path('/kaggle/input/datasets/<your-username>/<your-dataset-name>')
# ======================================================

records = [json.loads(l) for l in open(BASE / 'dataset.jsonl')]
existing_jpgs = set(f.name for f in BASE.glob('*.jpg'))
records = sorted(
    [r for r in records if r['frame'] in existing_jpgs],
    key=lambda r: r['frame']
)
print(f'Records: {len(records)}, images: {len(existing_jpgs)}')
print('Action distribution:', Counter(tuple(r['action']) for r in records))

In [ ]:
def action_vec_to_idx(a):
    return (int(a[0]) + 1) * 3 + (int(a[1]) + 1)

KEY_MAP   = {'w': (0,1), 's': (0,-1), 'a': (-1,0), 'd': (1,0), ' ': (0,0)}
N_ACTIONS = 9

# Hyperparameters
IMG_H, IMG_W = 128, 128
SEQ_LEN      = 8
BATCH_SIZE   = 8
LR           = 2e-4
EPOCHS       = 100
HIDDEN_DIM   = 512
LATENT_DIM   = 256
ACTION_DIM   = 64

transform = transforms.Compose([
    transforms.Resize((IMG_H, IMG_W)),
    transforms.ToTensor(),
    transforms.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])
])

In [ ]:
class WorldDataset(Dataset):
    def __init__(self, records, base_dir, seq_len=SEQ_LEN):
        self.records  = records
        self.base_dir = Path(base_dir)
        self.seq_len  = seq_len

    def __len__(self):
        return len(self.records) - self.seq_len

    def __getitem__(self, idx):
        frames, actions = [], []
        for i in range(idx, idx + self.seq_len + 1):
            r = self.records[i]
            img = Image.open(self.base_dir / r['frame']).convert('RGB')
            frames.append(transform(img))
            actions.append(action_vec_to_idx(r['action']))
        return torch.stack(frames), torch.tensor(actions, dtype=torch.long)

dataset = WorldDataset(records, BASE)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print(f'Samples: {len(dataset)}, batches: {len(loader)}')

In [ ]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.DEFAULT).features
        self.slice1 = nn.Sequential(*list(vgg)[:4]).eval()
        self.slice2 = nn.Sequential(*list(vgg)[4:9]).eval()
        self.slice3 = nn.Sequential(*list(vgg)[9:16]).eval()
        for p in self.parameters():
            p.requires_grad = False
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))

    def preprocess(self, x):
        x = (x * 0.5 + 0.5).clamp(0, 1)
        return (x - self.mean) / self.std

    def forward(self, pred, target):
        p, t = self.preprocess(pred), self.preprocess(target)
        p1, t1 = self.slice1(p), self.slice1(t)
        p2, t2 = self.slice2(p1), self.slice2(t1)
        p3, t3 = self.slice3(p2), self.slice3(t2)
        return F.mse_loss(p1,t1) + F.mse_loss(p2,t2) + F.mse_loss(p3,t3)

perc_loss_fn = PerceptualLoss().to(DEVICE)
print('Perceptual loss ready.')

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, 1, 1), nn.GroupNorm(8, ch), nn.SiLU(),
            nn.Conv2d(ch, ch, 3, 1, 1), nn.GroupNorm(8, ch)
        )
    def forward(self, x): return x + self.net(x)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,  32, 4, 2, 1), nn.SiLU(), ResBlock(32),
            nn.Conv2d(32, 64, 4, 2, 1), nn.SiLU(), ResBlock(64),
            nn.Conv2d(64,128, 4, 2, 1), nn.SiLU(), ResBlock(128),
            nn.Conv2d(128,256,4, 2, 1), nn.SiLU(), ResBlock(256),
            nn.Conv2d(256,512,4, 2, 1), nn.SiLU(),
            nn.Flatten(),
            nn.Linear(512*4*4, LATENT_DIM)
        )
    def forward(self, x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc  = nn.Linear(HIDDEN_DIM + LATENT_DIM, 512*4*4)
        self.net = nn.Sequential(
            ResBlock(512),
            nn.ConvTranspose2d(512,256, 4, 2, 1), nn.SiLU(), ResBlock(256),
            nn.ConvTranspose2d(256,128, 4, 2, 1), nn.SiLU(), ResBlock(128),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.SiLU(), ResBlock(64),
            nn.ConvTranspose2d( 64, 32, 4, 2, 1), nn.SiLU(), ResBlock(32),
            nn.ConvTranspose2d( 32,  3, 4, 2, 1), nn.Tanh(),
        )
    def forward(self, z): return self.net(self.fc(z).view(-1, 512, 4, 4))

class RSSM(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder      = Encoder()
        self.decoder      = Decoder()
        self.action_emb   = nn.Embedding(N_ACTIONS, ACTION_DIM)
        self.gru          = nn.GRUCell(LATENT_DIM + ACTION_DIM, HIDDEN_DIM)
        self.prior_fc     = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.SiLU(),
            nn.Linear(HIDDEN_DIM, LATENT_DIM * 2)
        )
        self.posterior_fc = nn.Sequential(
            nn.Linear(HIDDEN_DIM + LATENT_DIM, HIDDEN_DIM), nn.SiLU(),
            nn.Linear(HIDDEN_DIM, LATENT_DIM * 2)
        )

    def reparameterize(self, mu, logvar):
        return mu + (0.5 * logvar).exp() * torch.randn_like(mu)

    def forward(self, frames, actions):
        B, T1, C, H, W = frames.shape
        T = T1 - 1
        h = torch.zeros(B, HIDDEN_DIM, device=frames.device)
        recons, kl_total = [], torch.tensor(0., device=frames.device)
        for t in range(T):
            z_enc = self.encoder(frames[:, t])
            a_emb = self.action_emb(actions[:, t])
            mu_post, lv_post = self.posterior_fc(torch.cat([h, z_enc], -1)).chunk(2, -1)
            z_post = self.reparameterize(mu_post, lv_post)
            mu_prior, lv_prior = self.prior_fc(h).chunk(2, -1)
            kl = -0.5 * (1 + lv_post - lv_prior
                         - ((mu_post - mu_prior).pow(2) + lv_post.exp()) / (lv_prior.exp() + 1e-8))
            kl_total = kl_total + kl.mean()
            h = self.gru(torch.cat([z_post, a_emb], -1), h)
            recons.append(self.decoder(torch.cat([h, z_post], -1)))
        return torch.stack(recons, dim=1), kl_total / T

    @torch.no_grad()
    def imagine(self, frame0, actions_seq):
        h = torch.zeros(1, HIDDEN_DIM, device=frame0.device)
        frames_out = [frame0]
        for a_idx in actions_seq:
            a  = torch.tensor([a_idx], device=frame0.device)
            ae = self.action_emb(a)
            mu, lv = self.prior_fc(h).chunk(2, -1)
            z  = self.reparameterize(mu, lv)
            h  = self.gru(torch.cat([z, ae], -1), h)
            frames_out.append(self.decoder(torch.cat([h, z], -1)))
        return frames_out

model = RSSM().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

In [ ]:
optim     = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=EPOCHS)

W_MSE, W_PERC, W_KL = 1.0, 0.1, 0.01
history = {'loss': [], 'mse': [], 'perc': []}

for epoch in range(1, EPOCHS+1):
    model.train()
    ep = {'loss': 0., 'mse': 0., 'perc': 0.}
    for frames, actions in loader:
        frames  = frames.to(DEVICE)
        actions = actions.to(DEVICE)
        recons, kl = model(frames, actions)
        targets = frames[:, 1:]
        B, T, C, H, W = recons.shape
        r_flat = recons.reshape(B*T, C, H, W)
        t_flat = targets.reshape(B*T, C, H, W)
        mse_l  = F.mse_loss(r_flat, t_flat)
        perc_l = perc_loss_fn(r_flat, t_flat)
        loss   = W_MSE * mse_l + W_PERC * perc_l + W_KL * kl
        optim.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 100.)
        optim.step()
        ep['loss'] += loss.item()
        ep['mse']  += mse_l.item()
        ep['perc'] += perc_l.item()
    scheduler.step()
    for k in ep: history[k].append(ep[k] / len(loader))
    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{EPOCHS}  loss={history["loss"][-1]:.4f}  mse={history["mse"][-1]:.4f}  perc={history["perc"][-1]:.4f}')

print('Training complete.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12,3))
for ax, key in zip(axes, ['loss','mse','perc']):
    ax.plot(history[key]); ax.set_title(key); ax.grid(True)
plt.tight_layout(); plt.show()

torch.save(model.state_dict(), '/kaggle/working/worldmodel_v2.pt')
print('Model saved to /kaggle/working/worldmodel_v2.pt')

In [ ]:
# Reconstruction quality check
def unnorm(t):
    return (t * 0.5 + 0.5).clamp(0,1).permute(1,2,0).cpu().numpy()

model.eval()
sf, sa = dataset[0]
sf = sf.unsqueeze(0).to(DEVICE)
sa = sa.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    recons, _ = model(sf, sa)

T_show = min(6, SEQ_LEN)
fig, axes = plt.subplots(2, T_show, figsize=(T_show*3, 6))
for i in range(T_show):
    axes[0,i].imshow(unnorm(sf[0, i+1])); axes[0,i].set_title(f't+{i+1} real'); axes[0,i].axis('off')
    axes[1,i].imshow(unnorm(recons[0,i])); axes[1,i].set_title(f't+{i+1} pred'); axes[1,i].axis('off')
plt.suptitle('Top: ground truth  |  Bottom: model prediction')
plt.tight_layout(); plt.show()

In [ ]:
# Export trajectory GIF
def load_frame(idx=0):
    r = records[idx]
    img = Image.open(BASE / r['frame']).convert('RGB')
    return transform(img).unsqueeze(0).to(DEVICE)

action_demo = (
    [action_vec_to_idx([0, 1])] * 12 +
    [action_vec_to_idx([1, 0])] * 10 +
    [action_vec_to_idx([0,-1])] * 12 +
    [action_vec_to_idx([-1,0])] * 10
)

model.eval()
imagined = model.imagine(load_frame(0), action_demo)

frames_pil = []
for t in imagined:
    arr = (t.squeeze(0)*0.5+0.5).clamp(0,1)
    arr = (arr.permute(1,2,0).cpu().numpy()*255).astype(np.uint8)
    frames_pil.append(Image.fromarray(arr).resize((512,384), Image.LANCZOS))

frames_pil[0].save('/kaggle/working/trajectory.gif',
    save_all=True, append_images=frames_pil[1:], duration=80, loop=0)
print(f'GIF saved: {len(frames_pil)} frames')

# Download links
from IPython.display import FileLink, display
display(FileLink('worldmodel_v2.pt'))
display(FileLink('trajectory.gif'))